NB03_speed_eke_trends.py
========================
Compute geostrophic speed and EKE per site, then run Sen's slope +
Mann-Kendall trend analysis for all meander metrics.

Requires: NB02 outputs (monthly_metrics_{site}.csv)

Specifications from GRL manuscript (Sec 2.2–2.3):
  - Speed: monthly mean |V_geo| along meander centreline
  - EKE: 0.5*(u'² + v'²), anomalies relative to 1993–2025 monthly climatology
  - Trends: Sen's slope, Mann-Kendall test, modified MK when ACF(1) > 0.1
  - Significance: p < 0.05

Author: Xinlong Liu, IMAS, University of Tasmania

NCI Gadi setup (run once per session before launching this script):
    source /g/data/gv90/xl1657/venvs/cmems_py311/bin/activate
    pip install pymannkendall   # only needed once

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from netCDF4 import Dataset
import pymannkendall as mk
from statsmodels.tsa.stattools import acf
import warnings, gc

## 0. CONFIGURATION

In [ ]:
CONFIGURATION
# =============================================================================
BASE_DIR     = Path("/g/data/gv90/xl1657/cmems_adt")
ADT_FP       = BASE_DIR / "cmems_so30S_19930101_20250802_adt_sla_ugos_vgos_batch.nc"
PRODUCT_DIR  = BASE_DIR / "grl_meander_products"
OUT_DIR      = BASE_DIR / "grl_meander_products"

SITES = {
    "CP":   {"name": "Campbell Plateau",        "inner_lon": (150, -150), "inner_lat": (-57, -46), "wraps": True},
    "PAR":  {"name": "Pacific-Antarctic Ridge",  "inner_lon": (-150, -80), "inner_lat": (-60, -48), "wraps": False},
    "SEIR": {"name": "Southeast Indian Ridge",   "inner_lon": (130, 152), "inner_lat": (-56, -44), "wraps": False},
    "SWIR": {"name": "Southwest Indian Ridge",   "inner_lon": (15,  45),  "inner_lat": (-58, -44), "wraps": False},
}

BELT_LAT = (-60, -30)  # full belt for velocity extraction

## 1. COMPUTE SPEED AND EKE

In [ ]:
COMPUTE SPEED AND EKE
# =============================================================================
def compute_speed_eke(adt_fp, sites, product_dir):
    """
    For each site, compute monthly geostrophic speed and EKE.
    Speed is sampled at grid cells within the site inner domain.
    EKE = 0.5*(u'² + v'²) where u',v' are anomalies from monthly climatology.
    """
    print("Computing speed and EKE from geostrophic velocities...")

    with Dataset(str(adt_fp), "r") as src:
        lat_name = "latitude" if "latitude" in src.variables else "lat"
        lon_name = "longitude" if "longitude" in src.variables else "lon"
        lat = np.array(src.variables[lat_name][:], dtype=np.float64)
        lon = np.array(src.variables[lon_name][:], dtype=np.float64)
        time_raw = src.variables["time"]
        time_all = pd.DatetimeIndex(
            xr.coding.times.decode_cf_datetime(
                time_raw[:], time_raw.units,
                calendar=getattr(time_raw, "calendar", "standard"),
            )
        )

        nt = len(time_all)
        TIME_CHUNK = 30

        # For each site: compute daily spatial-mean speed, then aggregate to monthly
        for site_key, site in sites.items():
            print(f"\n  {site_key}: {site['name']}")

            ilon = site["inner_lon"]
            ilat_range = site["inner_lat"]
            wraps = site.get("wraps", False)
            jlat = np.where((lat >= ilat_range[0]) & (lat <= ilat_range[1]))[0]

            if not wraps:
                jlon = np.where((lon >= ilon[0]) & (lon <= ilon[1]))[0]
            else:
                jlon = np.where((lon >= ilon[0]) | (lon <= ilon[1]))[0]

            if len(jlat) == 0 or len(jlon) == 0:
                print(f"    WARNING: no grid points found. Skipping.")
                continue

            lat_s = slice(int(jlat[0]), int(jlat[-1]) + 1)

            # Determine if lon indices are contiguous
            if not wraps or len(jlon) == 0:
                lon_s = slice(int(jlon[0]), int(jlon[-1]) + 1)
                lon_contiguous = True
            else:
                gap = np.where(np.diff(jlon) > 1)[0]
                if len(gap) > 0:
                    split = gap[0] + 1
                    # low indices = west (-180..-150), high indices = east (150..180)
                    lon_s_1 = slice(int(jlon[:split][0]), int(jlon[:split][-1]) + 1)   # west
                    lon_s_2 = slice(int(jlon[split:][0]), int(jlon[split:][-1]) + 1)   # east
                    lon_contiguous = False
                else:
                    lon_s = slice(int(jlon[0]), int(jlon[-1]) + 1)
                    lon_contiguous = True

            # Accumulate daily spatial-mean speed and velocity components
            daily_speed = np.full(nt, np.nan)
            daily_u = np.full(nt, np.nan)
            daily_v = np.full(nt, np.nan)

            for cs in range(0, nt, TIME_CHUNK):
                ce = min(cs + TIME_CHUNK, nt)
                if lon_contiguous:
                    u = np.array(src.variables["ugos"][cs:ce, lat_s, lon_s], dtype=np.float32)
                    v = np.array(src.variables["vgos"][cs:ce, lat_s, lon_s], dtype=np.float32)
                else:
                    # Geographic order: east (150→180) then west (-180→-150)
                    u = np.concatenate([
                        np.array(src.variables["ugos"][cs:ce, lat_s, lon_s_2], dtype=np.float32),
                        np.array(src.variables["ugos"][cs:ce, lat_s, lon_s_1], dtype=np.float32),
                    ], axis=2)
                    v = np.concatenate([
                        np.array(src.variables["vgos"][cs:ce, lat_s, lon_s_2], dtype=np.float32),
                        np.array(src.variables["vgos"][cs:ce, lat_s, lon_s_1], dtype=np.float32),
                    ], axis=2)
                u = np.where(np.abs(u) > 50, np.nan, u)
                v = np.where(np.abs(v) > 50, np.nan, v)

                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", RuntimeWarning)
                    spd = np.sqrt(u**2 + v**2)
                    daily_speed[cs:ce] = np.nanmean(spd, axis=(1, 2))
                    daily_u[cs:ce] = np.nanmean(u, axis=(1, 2))
                    daily_v[cs:ce] = np.nanmean(v, axis=(1, 2))

            # Aggregate to monthly means
            df_daily = pd.DataFrame({
                "speed": daily_speed, "u": daily_u, "v": daily_v,
            }, index=time_all)
            df_monthly = df_daily.resample("MS").mean()
            df_monthly.index = df_monthly.index + pd.Timedelta(days=14)

            # Compute monthly climatology and anomalies for EKE
            u_clim = df_monthly["u"].groupby(df_monthly.index.month).transform("mean")
            v_clim = df_monthly["v"].groupby(df_monthly.index.month).transform("mean")
            u_anom = df_monthly["u"] - u_clim
            v_anom = df_monthly["v"] - v_clim
            eke = 0.5 * (u_anom**2 + v_anom**2)

            df_monthly["eke_m2_s2"] = eke.values

            # Load existing meander metrics and merge
            csv_fp = product_dir / f"monthly_metrics_{site_key}.csv"
            if csv_fp.exists():
                df_existing = pd.read_csv(csv_fp, index_col=0, parse_dates=True)
                df_existing["speed_m_s"] = np.nan
                df_existing["eke_m2_s2"] = np.nan

                for idx in df_monthly.index:
                    if idx in df_existing.index:
                        df_existing.loc[idx, "speed_m_s"] = df_monthly.loc[idx, "speed"]
                        df_existing.loc[idx, "eke_m2_s2"] = df_monthly.loc[idx, "eke_m2_s2"]

                df_existing.to_csv(csv_fp)
                print(f"    Updated: {csv_fp}")
                print(f"    Speed range: {df_monthly['speed'].min():.3f}–{df_monthly['speed'].max():.3f} m/s")
                print(f"    EKE range: {eke.min():.6f}–{eke.max():.6f} m²/s²")
            else:
                print(f"    WARNING: {csv_fp} not found. Run NB02 first.")

## 2. TREND ANALYSIS WITH PYMANNKENDALL

In [ ]:
TREND ANALYSIS WITH PYMANNKENDALL
# =============================================================================
def compute_trends(product_dir, sites, alpha=0.05):
    """
    Compute Sen's slope and Mann-Kendall significance for all metrics.
    Uses modified MK (Hamed & Rao 1998) when lag-1 autocorrelation > 0.1.

    Matches GRL manuscript Sec 2.3 exactly.
    """
    print("\nComputing trends with Sen's slope + Mann-Kendall...")

    metrics = {
        "center_lat":    ("Position",       "°/dec"),
        "width_km":      ("Width",          "km/dec"),
        "speed_m_s":     ("Speed",          "m s⁻¹/dec"),
        "eke_m2_s2":     ("EKE",            "m² s⁻²/dec"),
    }

    rows = []

    for site_key in sites:
        csv_fp = product_dir / f"monthly_metrics_{site_key}.csv"
        if not csv_fp.exists():
            print(f"  {site_key}: CSV not found, skipping.")
            continue

        df = pd.read_csv(csv_fp, index_col=0, parse_dates=True)
        print(f"\n  {site_key}: {sites[site_key]['name']}")

        for metric, (label, unit) in metrics.items():
            if metric not in df.columns:
                print(f"    {metric}: not in CSV, skipping.")
                continue

            y = df[metric].dropna()
            if len(y) < 24:
                print(f"    {metric}: too few points ({len(y)}), skipping.")
                continue

            # Remove seasonal cycle (manuscript uses monthly data)
            y_anom = y - y.groupby(y.index.month).transform("mean")
            series = y_anom.values

            # Check lag-1 autocorrelation
            acf_vals = acf(series, nlags=5, fft=True)
            lag1 = acf_vals[1]
            use_modified = abs(lag1) > 0.1

            # Run appropriate MK test
            if use_modified:
                result = mk.hamed_rao_modification_test(series)
                test_name = "Modified MK"
            else:
                result = mk.original_test(series)
                test_name = "Original MK"

            # Sen's slope per month → per decade
            slope_per_decade = result.slope * 12 * 10

            # R² from linear regression for context
            from scipy import stats as sp_stats
            x_idx = np.arange(len(series))
            r_val = sp_stats.pearsonr(x_idx, series)[0]
            r2 = r_val**2

            sig_str = "***" if result.p < 0.01 else ("*" if result.p < 0.05 else "ns")

            rows.append({
                "site": site_key,
                "site_name": sites[site_key]["name"],
                "metric": metric,
                "metric_label": label,
                "unit": unit,
                "slope_per_decade": slope_per_decade,
                "p_value": result.p,
                "significant": result.p < alpha,
                "test": test_name,
                "acf_lag1": lag1,
                "R2": r2,
                "n_obs": len(series),
            })

            print(f"    {label:20s}  slope={slope_per_decade:+.4e}/{unit}  "
                  f"p={result.p:.4f}  {sig_str}  [{test_name}]")

    trends_df = pd.DataFrame(rows)

    # Save
    out_fp = product_dir / "trend_results_mannkendall.csv"
    trends_df.to_csv(out_fp, index=False)
    print(f"\nTrend results saved to: {out_fp}")

    return trends_df

## 3. MAIN

In [ ]:
MAIN
# =============================================================================
if __name__ == "__main__":
    print("=" * 70)
    print("GRL Speed, EKE & Trend Analysis Pipeline")
    print("=" * 70)

    # Step 1: Compute speed and EKE
    compute_speed_eke(ADT_FP, SITES, PRODUCT_DIR)

    # Step 2: Trend analysis
    trends_df = compute_trends(PRODUCT_DIR, SITES)

    # Step 3: Print summary table
    print("\n" + "=" * 70)
    print("TREND SUMMARY FOR GRL MANUSCRIPT")
    print("=" * 70)
    if len(trends_df) > 0:
        sig = trends_df[trends_df["significant"]]
        print(f"\nSignificant trends (p < 0.05): {len(sig)} of {len(trends_df)}")
        for _, row in sig.iterrows():
            print(f"  {row['site']:5s} {row['metric_label']:10s} "
                  f"{row['slope_per_decade']:+.4e} {row['unit']}  "
                  f"p={row['p_value']:.4f}")

    print("\nPipeline complete.")